# 2. Factors behind visitation

This notebook tests whether verified NPS metadata adds explanatory information. The available fields are designation, latitude, longitude, and year. Acreage, population proximity, and travel accessibility are not present in the verified sources, so those questions remain unresolved rather than receiving unsupported proxies.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.analysis import load_recreation_visits, load_park_metadata, annual_visits, add_metadata, grouped_regression
raw = ROOT / 'data' / 'raw'
visits = load_recreation_visits(raw / 'Main_Data.csv')
metadata = load_park_metadata(raw / 'nps_parks_api.json')
annual = add_metadata(annual_visits(visits), metadata)
annual['log_visits'] = (annual.RecreationVisits + 1).map(__import__('numpy').log)
display(annual.groupby('Designation').RecreationVisits.sum().sort_values(ascending=False).head(12))

## Park-grouped regression
Repeated yearly observations create leakage if the same unit appears in both train and test. The helper therefore holds out whole units, uses log visits to reduce skew, and reports a modest descriptive benchmark.

In [ ]:
metrics = grouped_regression(annual)
pd.Series(metrics)

## Correlation and limitations
Correlation is not causation. A relationship involving coordinates or designation could reflect omitted factors such as attractions, roads, nearby population, fees, boundaries, or counting procedures. The low held-out R-squared means these available fields are not enough to explain much unit-level variation out of sample. Visits per acre is not calculated because acreage is unavailable and would not directly measure crowding even if it were available.

In [ ]:
numeric = annual[['RecreationVisits', 'Latitude', 'Longitude', 'Year']].corr(numeric_only=True)
display(numeric)
print('Unresolved variables: acreage, population proximity, and accessibility.')